<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

## Interpret a Passing KKT Result

**Use the shape of the clock score to distinguish a verified minimum from a stationary candidate.**

Keep the clock observations, calibration, and upper bound. A short comparison shows why convexity makes KKT conclusive here, and why changing the objective can remove that guarantee.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 1 · Retain calibration and impose an upper bound

The observed errors are $d_i=-2,-1,-1,0$ min at $t_i=0,1,2,3$ h, for $i=1,\ldots,4$. Choose initial correction $q$ in min and rate correction $r$ in min/h. The remaining errors are $y_i=d_i+q+rt_i$ and the squared-error objective is

$$
\phi(q,r)=\sum_{i=1}^{4}y_i^2.
$$

The original calibration requirement at one hour is $q+(1\,\mathrm h)r=1\,\mathrm{min}$. Using numerical values in these units, keep the equality residual $h(q,r)=q+r-1=0$. Let $q_{\max}$ be the upper limit on the initial correction, in minutes, and define the inequality residual $g(q,r)=q-q_{\max}\le0$.

Unless a comparison states otherwise, use $q_{\max}=1$. The equality-only solution $(1.5,-0.5)$ violates this upper bound. The next definition keeps physical errors, the objective, and feasibility checks separate.


In [ ]:
import numpy as np

# Fixed observations and their elapsed times
OBSERVED_ERRORS = np.array([-2.0, -1.0, -1.0, 0.0])  # min
OBSERVATION_TIMES = np.array([0.0, 1.0, 2.0, 3.0])   # h

# Calibration requirement: correction at 1 h must equal 1 min
CALIBRATION_TIME = 1.0        # h
REQUIRED_CORRECTION = 1.0     # min
CHECK_TOLERANCE = 1e-9        # numerical comparison setting


def simulate_clock(decision):
    correction, rate = np.asarray(decision, dtype=float)
    return OBSERVED_ERRORS + correction + rate * OBSERVATION_TIMES


def evaluate_clock(decision):
    decision = np.asarray(decision, dtype=float)
    remaining_errors = simulate_clock(decision)
    correction, rate = decision
    equality_residual = correction + CALIBRATION_TIME * rate - REQUIRED_CORRECTION
    gradient = 2 * np.array([np.sum(remaining_errors),
                             np.dot(OBSERVATION_TIMES, remaining_errors)])
    return {
        "decision": decision,
        "response": remaining_errors,
        "objective": float(np.sum(remaining_errors ** 2)),
        "gradient": gradient,
        "equality_residual": float(equality_residual),
        "feasible": bool(abs(equality_residual) <= CHECK_TOLERANCE),
    }


CORRECTION_LIMIT = 1.0  # min


def evaluate_bounded_clock(decision, correction_limit=CORRECTION_LIMIT):
    result = evaluate_clock(decision)
    inequality_residual = float(result["decision"][0] - correction_limit)
    result.update({
        "inequality_residual": inequality_residual,
        "feasible": bool(result["feasible"] and inequality_residual <= CHECK_TOLERANCE),
    })
    return result


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 2 · Compare feasible clock settings

Calibration gives $r=1-q$. Along that line, the score is

$$
\phi(q,1-q)=0.5+6(q-1.5)^2.
$$

This is a bowl with its lowest point at $q=1.5$. With the upper bound $q\le1$, that point is unavailable. The closest allowed correction is $q=1$, with score 2 min². Every smaller $q$ is farther from 1.5 and increases the squared term. This proves the minimum over the allowed line.

The code illustrates the distinction: $(0.8,0.2)$ is feasible but worse, $(1,0)$ is the minimum, and $(1.2,-0.2)$ has a smaller score but is rejected by the bound.


In [ ]:
for correction in (0.8, 1.0, 1.2):
    result = evaluate_bounded_clock([correction, 1 - correction], 1.0)
    status = "feasible" if result["feasible"] else "REJECT"
    print(f"q = {correction:.1f}: {status}, score = {result['objective']:.2f} min²")


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 3 · Explain why KKT is sufficient here

A **quadratic program (QP)** has a quadratic objective and affine constraints. Affine means linear terms plus constants. The clock score is a sum of squared affine errors, and its equality and bound are affine.

This objective is **convex**: along any straight change in the settings, its curve has a bowl shape. The feasible set is convex too: the segment between two feasible settings remains feasible. These properties make all four KKT conditions sufficient for a global minimum.

For these four observations, the objective is strictly convex, so the minimum is unique. The name QP alone is not enough: a quadratic objective can also curve downward.


<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
.katex .textbb { font-style: italic; }
</style>

### 4 · Change only the objective and observe the limit

Fix $q_{\max}=2$ and retain calibration in both comparisons. The original score $\phi$ is minimized at $(1.5,-0.5)$, with value 0.5 min². Now use the reversed objective $\psi=-\phi$. This would reward larger clock errors; it is only a demonstration of the limits of a KKT check.

The same settings satisfy KKT for the reversed objective with $\nu=-2$ and $\mu=0$. Yet their value is $-0.5$, while the feasible setting $(1,0)$ gives $-2$, which is smaller. The stationary point is at the top of the reversed bowl.

The code confirms the balanced slopes and the better feasible alternative. A KKT check must therefore be interpreted together with the objective's shape. The original convex clock problem has a guarantee that this reversed minimization problem does not.


In [ ]:
for correction in (1.0, 1.5):
    result = evaluate_bounded_clock([correction, 1 - correction], correction_limit=2.0)
    print(f"Original objective: q = {correction:.1f}, feasible = {result['feasible']}, "
          f"score = {result['objective']:.2f} min²")

reversed_candidate = evaluate_bounded_clock([1.5, -0.5], correction_limit=2.0)
reversed_gradient = -reversed_candidate["gradient"]
stationarity = reversed_gradient - 2 * np.array([1.0, CALIBRATION_TIME])
print("Reversed-objective stationarity:", stationarity)
print("Reversed candidate score:", -reversed_candidate["objective"])
print("Better feasible reversed score:", -evaluate_bounded_clock([1.0, 0.0], 2.0)["objective"])
